In [1]:
import pandas as pd 

# Utilise juste le nom du fichier
df_ml = pd.read_csv("C:\\Users\\user\\Projet3\\WorldCup2026\\notebooks\\df_overall_joueur_equipe.csv")

In [2]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler

# Ton mapping des chapeaux (rempli avec tes 48 équipes)
chapeaux_mapping = {
    'Chapeau 1': ['Germany', 'France', 'Spain', 'England', 'Portugal', 'Brazil', 'Argentina', 'Belgium', 'Netherlands', 'Norway'], # Les favoris
    'Chapeau 2': ['Japan', 'Senegal', 'Ivory Coast', 'Canada', 'Mexico', 'Colombia', 'Turkey', 'Croatia', 'Czechia', 'Switzerland', 'Austria', 'Uruguay', 'United States'],
    'Chapeau 3': ['Egypt', 'Tunisia', 'Australia', 'Congo DR', 'Bosnia-Herzegovina', 'Ecuador', 'Ghana', 'Paraguay', 'Algeria', 'South Korea', 'Scotland', 'South Africa','Morocco'],
    'Chapeau 4': ['Jordan', 'Saudi Arabia', 'Uzbekistan', "Iran", 'Qatar', "Haiti", "Iraq", "Curaçao", "Cape Verde Islands", "Panama", "New Zealand", "Tunisia"]  # Les outsiders
}

In [3]:
def assigner_niveau(equipe):
    for niveau, liste in chapeaux_mapping.items():
        if equipe in liste:
            if niveau == 'Chapeau 1': return 32
            if niveau == 'Chapeau 2': return 16
            if niveau == 'Chapeau 3': return 8
            if niveau == 'Chapeau 4': return 2
    return 2.5 

# Application du niveau
df_ml['niveau_equipe'] = df_ml['equipe_nationale'].apply(assigner_niveau)

# Définition des features (les variables de jeu)
features_ml = [col for col in df_ml.columns if col.endswith('_par_90') and col != 'buts_marques_par_90']
features_ml += ['overall', 'moyenne_equipe']

X = df_ml[features_ml]
y = df_ml['buts_marques_par_90']

In [4]:
from sklearn.model_selection import train_test_split

# On définit X et y
X = df_ml[features_ml]
y = df_ml['buts_marques_par_90']

# LE SPLIT : On sépare les données ici
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# L'ENTRAÎNEMENT : On entraîne uniquement sur le Train
model_buts = RandomForestRegressor(n_estimators=200, random_state=42)
model_buts.fit(X_train, y_train)

# ÉVALUATION : On regarde si le modèle est bon sur le Test
score = model_buts.score(X_test, y_test)
print(f"Précision du modèle (R²) : {score:.2f}")

Précision du modèle (R²) : 0.91


In [5]:
# On prédit sur tout le dataset (X complet)
pred_brute = model_buts.predict(X)

# Ton boost contextuel
boost = 1 + (df_ml['niveau_equipe'] * 0.15)
df_ml['buts_predits_final'] = pred_brute * boost

In [6]:
# 1. Vérification des colonnes nécessaires
cols_to_scale = ['buts_marques_par_90', 'buts_predits_final']

# 2. Normalisation sécurisée
scaler = MinMaxScaler()
df_ml[['buts_norm', 'pred_norm']] = scaler.fit_transform(df_ml[cols_to_scale])

# 3. Calcul du score final avec ton boost 40/60
df_ml['score_final'] = (df_ml['buts_norm'] * 0.4) + (df_ml['pred_norm'] * 0.6)

# Créer une liste des joueurs intouchables
top_joueurs_historiques1 = ['Kylian Mbappé', 'Harry Kane', 'Lamine Yamal', 'Vinicius Junior', 'Mikel Oyarzabal', 'Lautaro Martinez']  
top_joueurs_historiques2 = ['Erling Haaland']  

# Appliquer un boost manuel de 15% à leur score final
df_ml['score_final'] = df_ml.apply(
    lambda row: row['score_final'] * 1.15 if row['nom_joueur'] in top_joueurs_historiques1 else row['score_final'], 
    axis=1
)
# Appliquer un boost manuel de 5% à leur score final
df_ml['score_final'] = df_ml.apply(
    lambda row: row['score_final'] * 1.05 if row['nom_joueur'] in top_joueurs_historiques2 else row['score_final'], 
    axis=1
)

# On applique un malus de 10% (multiplication par 0.9)
def appliquer_ajustement(row):
    nom = row['nom_joueur']
    
    # Liste des joueurs à "sanctionner"
    joueurs_malus = ['Cristiano Ronaldo', 'Deniz Undav', 'Raphinha', 'Borja Iglesias']
    
    if nom in joueurs_malus:
        return row['score_final'] * 0.90  # Malus de 10%
    return row['score_final']

# Appliquer la fonction sur tout ton dataframe
df_ml['score_final'] = df_ml.apply(appliquer_ajustement, axis=1)

# 4. Affichage propre
top_final = df_ml.sort_values(by='score_final', ascending=False).head(10)

print("CLASSEMENT - TOP 10")
# On vérifie bien que les colonnes existent avant d'imprimer
print(top_final[['nom_joueur', 'equipe_nationale', 'buts_marques_par_90', 'score_final']].to_string(index=False))

CLASSEMENT - TOP 10
       nom_joueur equipe_nationale  buts_marques_par_90  score_final
       Harry Kane          England             1.358491     1.150000
    Kylian Mbappé           France             1.055866     0.922886
     Lionel Messi        Argentina             1.115385     0.767081
   Erling Haaland           Norway             0.815710     0.727233
  Ousmane Dembélé           France             0.857143     0.680845
Cristiano Ronaldo         Portugal             0.964040     0.669803
     Lamine Yamal            Spain             0.628546     0.624547
    Ferrán Torres            Spain             0.711111     0.588809
 Lautaro Martínez        Argentina             0.693878     0.585856
         Raphinha           Brazil             0.821629     0.585013


In [12]:
# 1. On définit le multiplicateur selon le chapeau
# Chapeau 1 (4) -> 1.0 (référence)
# Chapeau 2 (3) -> 0.85 (légère baisse de productivité)
# Chapeau 3 (2) -> 0.70
# Chapeau 4 (1) -> 0.55
mapping_multiplicateur = {4: 1.0, 3: 0.80, 2: 0.70, 1: 0.55}

# 2. Création de la colonne "Buts Projetés"
# On prend la prédiction du modèle et on applique le coefficient de réalité du chapeau
df_ml['buts_projectes'] = df_ml['buts_predits_final'] * df_ml['niveau_equipe'].map(lambda x: mapping_multiplicateur.get(x, 0.9))

# 3. On ajoute le bonus pour tes joueurs "Elite" (Mbappé, Kane, etc.)
# Si tu veux qu'ils soient au-dessus des stats collectives
top_joueurs = ['Lautaro Martinez', 'Ferran Torres']
top_joueurs2 = ['Kylian Mbappé', 'Lionel Messi']
top_joueurs3 = ['Erling Haaland', 'Ousmane Dembélé', 'Raphinha', 'Lamine Yamal', 'Harry Kane']
top_joueurs4 = ['Cristiano Ronaldo']
malus_joueurs = ['Deniz Undav', "Donyell Malen"]
df_ml.loc[df_ml['nom_joueur'].isin(top_joueurs), 'buts_projectes'] *= 1.95
df_ml.loc[df_ml['nom_joueur'].isin(top_joueurs2), 'buts_projectes'] *= 1.2
df_ml.loc[df_ml['nom_joueur'].isin(top_joueurs3), 'buts_projectes'] *= 1.04
df_ml.loc[df_ml['nom_joueur'].isin(top_joueurs4), 'buts_projectes'] *= 1.002
df_ml.loc[df_ml['nom_joueur'].isin(malus_joueurs), 'buts_projectes'] *= 0.89

# 4. Affichage du résultat en buts réels
top_10 = df_ml.sort_values(by='buts_projectes', ascending=False).head(10)
print(top_10[['nom_joueur', 'buts_projectes']])

            nom_joueur  buts_projectes
8           Harry Kane        6.125500
1        Kylian Mbappé        5.791134
43        Lionel Messi        5.167355
6       Erling Haaland        4.618852
74   Cristiano Ronaldo        4.528255
5      Ousmane Dembélé        4.374265
14            Raphinha        4.166265
19    Lautaro Martínez        3.745466
117      Ferrán Torres        3.724640
13        Lamine Yamal        3.655010


In [13]:
df_ml.to_csv("df_joueurs_niveau_equipe.csv", index=False)